# 03 – Evaluation protocol and first model comparison
**Project 2 – Ensemble learning for disease prediction**

What this notebook does
1. Loads the available datasets (same loader as the EDA).
2. Runs **8 models** (Decision Tree, Bagging, Random Forest, Extra Trees, AdaBoost, Gradient Boosting, XGBoost, Stacking)
   under **two protocols**:
   * `naive` – repeated stratified 5-fold CV on the data as downloaded (duplicates kept);
   * `clean` – exact duplicate rows removed first, then the same CV (no test row has a copy in train).
3. Reports Accuracy, Precision, Recall, F1 (required) + ROC-AUC and AUPRC (complementary) and confusion matrices.
4. Measures how much the naive protocol **overestimates** the scores, and ranks the models.

Fixed hyper-parameters here (defaults documented in `models.py`); tuning with an equal budget comes next.
Imputation and encoding are inside each fold (no leakage). Set `QUICK = True` for a fast test run.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
from ensemble_disease.config import load_config, set_seed
from ensemble_disease import eda, protocol
from ensemble_disease.models import get_models

cfg = load_config(); SEED = cfg["seed"]; set_seed(SEED)
N_SPLITS, N_REPEATS = cfg["cv"]["n_splits"], cfg["cv"]["n_repeats"]
QUICK = False          # True = 1 repeat, 3 models, to check that everything runs (~1 min)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

## 1. Data and models

In [ ]:
data, errors = eda.load_all()
print(f"{len(data)} datasets:", list(data))
for k, v in errors.items(): print(f"[NOT LOADED] {k}: {v[:120]}")

models = get_models(SEED)
if QUICK:
    models = {k: models[k] for k in ["DecisionTree", "RandomForest", "AdaBoost"]}
    N_REPEATS = 1
print("\nmodels:", list(models), "| CV:", N_SPLITS, "folds x", N_REPEATS, "repeats")

## 2. Run the cross-validation (both protocols)
This is the long cell: a few minutes for 7-9 datasets with 8 models. Progress is printed.

In [ ]:
results = protocol.run_all(data, models, protocols=("naive", "clean"),
                           n_splits=N_SPLITS, n_repeats=N_REPEATS, seed=SEED)
protocol.RESULTS.mkdir(exist_ok=True)
results.to_csv(protocol.RESULTS / "cv_results_long.csv", index=False)
print(results.shape, "-> results/cv_results_long.csv  (one row per dataset x protocol x model x fold)")

## 3. Results under the clean protocol (our reference)

In [ ]:
f1_clean = protocol.summarize(results, "f1", "clean")
display(f1_clean.style.background_gradient(cmap="viridis", axis=1).format("{:.3f}"))
protocol.plot_heatmap(f1_clean, "Mean F1 - clean protocol", "f1_clean_heatmap.png"); plt.show()

In [ ]:
for m in ["accuracy", "precision", "recall"]:
    print(f"\n=== {m} (clean) ===")
    display(protocol.summarize(results, m, "clean"))
print("\n=== ROC-AUC (clean) ===");  display(protocol.summarize(results, "roc_auc", "clean"))
print("\n=== AUPRC (clean) ===");    display(protocol.summarize(results, "auprc", "clean"))
print("\n=== std of F1 over folds (clean) ==="); display(protocol.summarize(results, "f1", "clean", std=True))

## 4. How much does duplicate leakage inflate the scores?
`naive - clean` (F1). A large positive value means the naive protocol is too optimistic for that dataset.

In [ ]:
infl = protocol.inflation(results, "f1")
display(infl.style.background_gradient(cmap="coolwarm", axis=None, vmin=-0.1, vmax=0.1).format("{:+.3f}"))
protocol.plot_heatmap(infl, "F1 inflation (naive - clean)", "f1_inflation_heatmap.png", center=0, cmap="coolwarm"); plt.show()
print("rows used per dataset:")
display(results.groupby(["dataset", "protocol"]).n_rows_used.first().unstack())

## 5. Ranking of the models (clean protocol, F1)
Mean rank over datasets (1 = best). This table is the input of the Friedman / Nemenyi tests (next step).

In [ ]:
ranks = protocol.mean_ranks(f1_clean)
display(ranks.to_frame("mean_rank"))
fig, ax = plt.subplots(figsize=(7, 3.5)); ranks[::-1].plot.barh(ax=ax, color="#3182bd")
ax.set_xlabel("mean rank (lower = better)"); ax.set_title("Model ranking over datasets (F1, clean)")
fig.savefig(protocol.RESULTS / "model_ranking.png", dpi=150, bbox_inches="tight"); plt.show()

## 6. Confusion matrices (best model per dataset, summed over all folds, clean protocol)

In [ ]:
cm = protocol.confusion_totals(results, "clean")
best = f1_clean.idxmax(axis=1)
rows = []
for ds, m in best.items():
    r = cm.loc[(ds, m)]
    rows.append(dict(dataset=ds, best_model=m, TN=r.tn, FP=r.fp, FN=r.fn, TP=r.tp,
                     FN_rate=round(r.fn / max(1, r.fn + r.tp), 3), FP_rate=round(r.fp / max(1, r.fp + r.tn), 3)))
cm_best = pd.DataFrame(rows).set_index("dataset"); display(cm_best)
cm_best.to_csv(protocol.RESULTS / "confusion_best_models.csv")

n = len(best); fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 2.8))
for ax, (ds, m) in zip(np.atleast_1d(axes), best.items()):
    r = cm.loc[(ds, m)]; M = np.array([[r.tn, r.fp], [r.fn, r.tp]])
    ax.imshow(M, cmap="Blues")
    for i in range(2):
        for j in range(2): ax.text(j, i, int(M[i, j]), ha="center", va="center", fontsize=9)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["pred 0", "pred 1"], fontsize=7)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["true 0", "true 1"], fontsize=7)
    ax.set_title(f"{ds.split('_')[0]}\n{m}", fontsize=8)
fig.tight_layout(); fig.savefig(protocol.RESULTS / "confusion_best_models.png", dpi=150); plt.show()

## 7. Quick reading guide (to write in the report)
* Compare the **ranking** with the EDA groups: on the *easy* datasets (D12, D16, D8) the models should be nearly tied;
  on the *hard* ones (D5, D6, D11) the differences are the informative ones.
* **FN_rate** matters medically: a missed sick patient is worse than a false alarm. Recall deserves as much attention as accuracy.
* The *inflation* table quantifies the duplicate-leakage critique of the reference article (largest expected on D8).
* A model that wins by 0.005 F1 with a std of 0.03 over folds is not really better: this is why the Friedman / Nemenyi tests come next.

## Next
Send back the tables of sections 3, 4, 5 and 6 (or `results/cv_results_long.csv`).